# Compare Raw 3D LLT vs Chebyshev Fit

This notebook compares the raw 3D LLT solver outputs with the Chebyshev polynomial fit.

- **Raw LLT**: Direct outputs from the 3D lifting line solver at sampling points
- **Chebyshev Fit**: Polynomial approximation (degree 25) fitted to the raw data

In [3]:
import numpy as np
import plotly.graph_objects as go
from pathlib import Path

# Path to the export directory (update this to the latest timestamp)
export_dir = Path("20260217_163210")

print(f"Loading artifacts from: {export_dir}")

Loading artifacts from: 20260217_163210


## Load Wing Data

In [4]:
# Load raw LLT outputs
wing_CL_raw = np.load(export_dir / "wing_raw_CL.npy")
wing_CD_raw = np.load(export_dir / "wing_raw_CD.npy")
wing_CM_raw = np.load(export_dir / "wing_raw_CM.npy")

# Load sampling grid
alpha_grid = np.load(export_dir / "sampling_alpha_grid.npy")
Re_grid = np.load(export_dir / "sampling_Re_grid.npy")

# Load Chebyshev coefficients
wing_phi_CL = np.load(export_dir / "wing_cheby_phi_CL.npy")
wing_phi_CD = np.load(export_dir / "wing_cheby_phi_CD.npy")
wing_phi_CM = np.load(export_dir / "wing_cheby_phi_CM.npy")

print(f"Wing raw CL shape: {wing_CL_raw.shape}")
print(f"Wing raw CD shape: {wing_CD_raw.shape}")
print(f"Alpha grid shape: {alpha_grid.shape}")
print(f"Re grid shape: {Re_grid.shape}")
print(f"Chebyshev coefficients shape: {wing_phi_CL.shape}")

Wing raw CL shape: (1024,)
Wing raw CD shape: (1024,)
Alpha grid shape: (1024,)
Re grid shape: (1024,)
Chebyshev coefficients shape: (676, 1)


## Chebyshev Evaluation Function

In [5]:
def chebyshev_basis(alpha_scaled, Re_scaled, degree=25):
    """Compute 2D Chebyshev tensor product basis."""
    B = alpha_scaled.shape[0]
    
    T_alpha = np.zeros((B, degree + 1))
    T_Re = np.zeros((B, degree + 1))
    
    T_alpha[:, 0] = 1
    T_Re[:, 0] = 1
    
    if degree >= 1:
        T_alpha[:, 1] = alpha_scaled
        T_Re[:, 1] = Re_scaled
    
    for n in range(2, degree + 1):
        T_alpha[:, n] = 2 * alpha_scaled * T_alpha[:, n - 1] - T_alpha[:, n - 2]
        T_Re[:, n] = 2 * Re_scaled * T_Re[:, n - 1] - T_Re[:, n - 2]
    
    return (T_alpha[:, :, None] * T_Re[:, None, :]).reshape(B, -1)

def scale_to_domain(x, min_val, max_val):
    """Scale x from [min_val, max_val] to [-1, 1]."""
    return 2 * (x - min_val) / (max_val - min_val) - 1

def evaluate_chebyshev(alpha, Re, phi, alpha_range, Re_range, degree=25):
    """Evaluate Chebyshev fit at given (alpha, Re) points."""
    alpha_scaled = scale_to_domain(alpha.flatten(), alpha_range[0], alpha_range[1])
    Re_scaled = scale_to_domain(Re.flatten(), Re_range[0], Re_range[1])
    
    X = chebyshev_basis(alpha_scaled, Re_scaled, degree)
    return (X @ phi).flatten()

# Training ranges (from conf/test.yaml)
alpha_range = (-30.0, 30.0)
Re_range = (100.0, 100000.0)

print(f"Training ranges: α=[{alpha_range[0]}, {alpha_range[1]}]°, Re=[{Re_range[0]}, {Re_range[1]}]")

Training ranges: α=[-30.0, 30.0]°, Re=[100.0, 100000.0]


## Evaluate Chebyshev Fit at Raw Sample Points

In [6]:
# Evaluate Chebyshev fit at the same sample points as raw data
wing_CL_fit = evaluate_chebyshev(alpha_grid, Re_grid, wing_phi_CL, alpha_range, Re_range)
wing_CD_fit = evaluate_chebyshev(alpha_grid, Re_grid, wing_phi_CD, alpha_range, Re_range)
wing_CM_fit = evaluate_chebyshev(alpha_grid, Re_grid, wing_phi_CM, alpha_range, Re_range)

# Compute errors
CL_error = np.abs(wing_CL_raw.flatten() - wing_CL_fit)
CD_error = np.abs(wing_CD_raw.flatten() - wing_CD_fit)
CM_error = np.abs(wing_CM_raw.flatten() - wing_CM_fit)

print(f"Wing CL - Mean absolute error: {CL_error.mean():.6f}, Max: {CL_error.max():.6f}")
print(f"Wing CD - Mean absolute error: {CD_error.mean():.6f}, Max: {CD_error.max():.6f}")
print(f"Wing CM - Mean absolute error: {CM_error.mean():.6f}, Max: {CM_error.max():.6f}")

Wing CL - Mean absolute error: 0.014945, Max: 0.094005
Wing CD - Mean absolute error: 0.002493, Max: 0.012092
Wing CM - Mean absolute error: 0.001999, Max: 0.011261


## Plot Wing CL: Raw vs Chebyshev Fit

In [7]:
fig = go.Figure()

# Scatter plot of raw LLT data
fig.add_trace(go.Scatter3d(
    x=Re_grid.flatten(),
    y=alpha_grid.flatten(),
    z=wing_CL_raw.flatten(),
    mode='markers',
    marker=dict(size=3, color='blue', opacity=0.6),
    name='Raw LLT',
    hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CL: %{z:.4f}<extra></extra>'
))

# Surface plot of Chebyshev fit (on a finer grid for smooth surface)
n_grid = 50
Re_fine = np.linspace(Re_grid.min(), Re_grid.max(), n_grid)
alpha_fine = np.linspace(alpha_grid.min(), alpha_grid.max(), n_grid)
RE_mesh, AA_mesh = np.meshgrid(Re_fine, alpha_fine)

CL_surface = evaluate_chebyshev(AA_mesh, RE_mesh, wing_phi_CL, alpha_range, Re_range)
CL_surface = CL_surface.reshape(n_grid, n_grid)

fig.add_trace(go.Surface(
    x=Re_fine,
    y=alpha_fine,
    z=CL_surface,
    colorscale='Viridis',
    name='Chebyshev Fit',
    showscale=True,
    opacity=0.7,
    hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CL: %{z:.4f}<extra></extra>'
))

fig.update_layout(
    title="Wing CL: Raw 3D LLT vs Chebyshev Fit",
    scene=dict(
        xaxis_title='Reynolds Number',
        yaxis_title='Angle of Attack [deg]',
        zaxis_title='CL',
    ),
    width=1000,
    height=700,
    showlegend=True
)

fig.show()

## Plot Wing CD: Raw vs Chebyshev Fit

In [8]:
fig = go.Figure()

# Scatter plot of raw LLT data
fig.add_trace(go.Scatter3d(
    x=Re_grid.flatten(),
    y=alpha_grid.flatten(),
    z=wing_CD_raw.flatten(),
    mode='markers',
    marker=dict(size=3, color='red', opacity=0.6),
    name='Raw LLT',
    hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CD: %{z:.4f}<extra></extra>'
))

# Surface plot of Chebyshev fit
CD_surface = evaluate_chebyshev(AA_mesh, RE_mesh, wing_phi_CD, alpha_range, Re_range)
CD_surface = CD_surface.reshape(n_grid, n_grid)

fig.add_trace(go.Surface(
    x=Re_fine,
    y=alpha_fine,
    z=CD_surface,
    colorscale='Plasma',
    name='Chebyshev Fit',
    showscale=True,
    opacity=0.7,
    hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CD: %{z:.4f}<extra></extra>'
))

fig.update_layout(
    title="Wing CD: Raw 3D LLT vs Chebyshev Fit",
    scene=dict(
        xaxis_title='Reynolds Number',
        yaxis_title='Angle of Attack [deg]',
        zaxis_title='CD',
    ),
    width=1000,
    height=700,
    showlegend=True
)

fig.show()

## Load Elevator Data

In [9]:
# Check if elevator data exists
elevator_files_exist = (export_dir / "elevator_raw_CL.npy").exists()

if elevator_files_exist:
    # Load raw LLT outputs
    elev_CL_raw = np.load(export_dir / "elevator_raw_CL.npy")
    elev_CD_raw = np.load(export_dir / "elevator_raw_CD.npy")
    elev_CM_raw = np.load(export_dir / "elevator_raw_CM.npy")
    
    # Load Chebyshev coefficients
    elev_phi_CL = np.load(export_dir / "elevator_cheby_phi_CL.npy")
    elev_phi_CD = np.load(export_dir / "elevator_cheby_phi_CD.npy")
    elev_phi_CM = np.load(export_dir / "elevator_cheby_phi_CM.npy")
    
    print(f"Elevator raw CL shape: {elev_CL_raw.shape}")
    print(f"Elevator Chebyshev coefficients shape: {elev_phi_CL.shape}")
    
    # Evaluate Chebyshev fit
    elev_CL_fit = evaluate_chebyshev(alpha_grid, Re_grid, elev_phi_CL, alpha_range, Re_range)
    elev_CD_fit = evaluate_chebyshev(alpha_grid, Re_grid, elev_phi_CD, alpha_range, Re_range)
    elev_CM_fit = evaluate_chebyshev(alpha_grid, Re_grid, elev_phi_CM, alpha_range, Re_range)
    
    # Compute errors
    elev_CL_error = np.abs(elev_CL_raw.flatten() - elev_CL_fit)
    elev_CD_error = np.abs(elev_CD_raw.flatten() - elev_CD_fit)
    elev_CM_error = np.abs(elev_CM_raw.flatten() - elev_CM_fit)
    
    print(f"\nElevator CL - Mean absolute error: {elev_CL_error.mean():.6f}, Max: {elev_CL_error.max():.6f}")
    print(f"Elevator CD - Mean absolute error: {elev_CD_error.mean():.6f}, Max: {elev_CD_error.max():.6f}")
    print(f"Elevator CM - Mean absolute error: {elev_CM_error.mean():.6f}, Max: {elev_CM_error.max():.6f}")
else:
    print("Elevator data not found in export directory")

Elevator raw CL shape: (1024,)
Elevator Chebyshev coefficients shape: (676, 1)

Elevator CL - Mean absolute error: 0.015035, Max: 0.096677
Elevator CD - Mean absolute error: 0.002250, Max: 0.008078
Elevator CM - Mean absolute error: 0.002093, Max: 0.012727


## Plot Elevator CL: Raw vs Chebyshev Fit

In [10]:
if elevator_files_exist:
    fig = go.Figure()
    
    # Scatter plot of raw LLT data
    fig.add_trace(go.Scatter3d(
        x=Re_grid.flatten(),
        y=alpha_grid.flatten(),
        z=elev_CL_raw.flatten(),
        mode='markers',
        marker=dict(size=3, color='green', opacity=0.6),
        name='Raw LLT',
        hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CL: %{z:.4f}<extra></extra>'
    ))
    
    # Surface plot of Chebyshev fit
    elev_CL_surface = evaluate_chebyshev(AA_mesh, RE_mesh, elev_phi_CL, alpha_range, Re_range)
    elev_CL_surface = elev_CL_surface.reshape(n_grid, n_grid)
    
    fig.add_trace(go.Surface(
        x=Re_fine,
        y=alpha_fine,
        z=elev_CL_surface,
        colorscale='Viridis',
        name='Chebyshev Fit',
        showscale=True,
        opacity=0.7,
        hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CL: %{z:.4f}<extra></extra>'
    ))
    
    fig.update_layout(
        title="Elevator CL: Raw 3D LLT vs Chebyshev Fit",
        scene=dict(
            xaxis_title='Reynolds Number',
            yaxis_title='Angle of Attack [deg]',
            zaxis_title='CL',
        ),
        width=1000,
        height=700,
        showlegend=True
    )
    
    fig.show()
else:
    print("Skipping elevator CL plot (no data)")

## Plot Elevator CD: Raw vs Chebyshev Fit

In [11]:
if elevator_files_exist:
    fig = go.Figure()
    
    # Scatter plot of raw LLT data
    fig.add_trace(go.Scatter3d(
        x=Re_grid.flatten(),
        y=alpha_grid.flatten(),
        z=elev_CD_raw.flatten(),
        mode='markers',
        marker=dict(size=3, color='orange', opacity=0.6),
        name='Raw LLT',
        hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CD: %{z:.4f}<extra></extra>'
    ))
    
    # Surface plot of Chebyshev fit
    elev_CD_surface = evaluate_chebyshev(AA_mesh, RE_mesh, elev_phi_CD, alpha_range, Re_range)
    elev_CD_surface = elev_CD_surface.reshape(n_grid, n_grid)
    
    fig.add_trace(go.Surface(
        x=Re_fine,
        y=alpha_fine,
        z=elev_CD_surface,
        colorscale='Plasma',
        name='Chebyshev Fit',
        showscale=True,
        opacity=0.7,
        hovertemplate='Re: %{x:.0f}<br>α: %{y:.2f}°<br>CD: %{z:.4f}<extra></extra>'
    ))
    
    fig.update_layout(
        title="Elevator CD: Raw 3D LLT vs Chebyshev Fit",
        scene=dict(
            xaxis_title='Reynolds Number',
            yaxis_title='Angle of Attack [deg]',
            zaxis_title='CD',
        ),
        width=1000,
        height=700,
        showlegend=True
    )
    
    fig.show()
else:
    print("Skipping elevator CD plot (no data)")

## Summary Statistics

In [12]:
import pandas as pd

summary_data = [
    ["Wing CL", CL_error.mean(), CL_error.max(), CL_error.std()],
    ["Wing CD", CD_error.mean(), CD_error.max(), CD_error.std()],
    ["Wing CM", CM_error.mean(), CM_error.max(), CM_error.std()],
]

if elevator_files_exist:
    summary_data.extend([
        ["Elevator CL", elev_CL_error.mean(), elev_CL_error.max(), elev_CL_error.std()],
        ["Elevator CD", elev_CD_error.mean(), elev_CD_error.max(), elev_CD_error.std()],
        ["Elevator CM", elev_CM_error.mean(), elev_CM_error.max(), elev_CM_error.std()],
    ])

summary_df = pd.DataFrame(
    summary_data,
    columns=["Coefficient", "Mean Abs Error", "Max Abs Error", "Std Error"]
)

print("\n" + "="*70)
print("CHEBYSHEV FIT ACCURACY SUMMARY")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)
print(f"\nNumber of sample points: {len(alpha_grid.flatten())}")
print(f"Chebyshev degree: 25 (676 coefficients per output)")
print(f"Alpha range: [{alpha_grid.min():.1f}, {alpha_grid.max():.1f}]°")
print(f"Re range: [{Re_grid.min():.0f}, {Re_grid.max():.0f}]")


CHEBYSHEV FIT ACCURACY SUMMARY
Coefficient  Mean Abs Error  Max Abs Error  Std Error
    Wing CL        0.014945       0.094005   0.016854
    Wing CD        0.002493       0.012092   0.002056
    Wing CM        0.001999       0.011261   0.002005
Elevator CL        0.015035       0.096677   0.017584
Elevator CD        0.002250       0.008078   0.001897
Elevator CM        0.002093       0.012727   0.002085

Number of sample points: 1024
Chebyshev degree: 25 (676 coefficients per output)
Alpha range: [-30.0, 30.0]°
Re range: [160, 99940]
